# Create SWORD Example Reach Matches

This notebook rebuilds the per-example SWORD reach matches in a clean way using the reusable library code in `gauge_sword_match.reach_paths`.

It is designed to stay compatible with the existing downstream bbox workflow by writing:

- `hierarchy_example_reach_lists.parquet`
- `hierarchy_example_reach_summary.csv`

Optional QA exports are also included for routed reach paths and routed reach geometries.

In [1]:
from __future__ import annotations

import ast
import re
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd


def resolve_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "gauge_sword_match").exists():
            return candidate
    raise FileNotFoundError("Could not find SWORD_gauge_match project root from the current working directory.")


PROJECT_ROOT = resolve_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from gauge_sword_match.reach_paths import build_example_reach_paths
from gauge_sword_match.sword_io import load_reaches, scan_sword_parquet_dir

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /Users/6256481/Code/river-hierarchy/SWORD_gauge_match
SRC_DIR: /Users/6256481/Code/river-hierarchy/SWORD_gauge_match/src


In [8]:
# Parameters

EXAMPLES_GPKG_PATH = PROJECT_ROOT / "outputs" / "hierarchy_examples_final.gpkg"
EXAMPLES_LAYER = "hierarchy_examples_final"

MAIN_CROSSWALK_BEST_PATH = PROJECT_ROOT / "outputs" / "crosswalk_best.parquet"
GRDC_CROSSWALK_BEST_PATH = PROJECT_ROOT / "outputs" / "grdc" / "crosswalk_best.parquet"

HYDRO_QUALITY_XLSX_PATH = PROJECT_ROOT / "outputs" / "mask_hydro_quality.xlsx"
HYDRO_QUALITY_SHEET = "Hydrograph_quality"

SWORD_PARQUET_DIR = Path("/Volumes/PhD/SWORD/v17c/beta/parquet")

OUTPUT_REACH_LISTS_PATH = PROJECT_ROOT / "outputs" / "hierarchy_example_reach_lists.parquet"
OUTPUT_REACH_SUMMARY_PATH = PROJECT_ROOT / "outputs" / "hierarchy_example_reach_summary.csv"

# Optional QA exports
OUTPUT_REACH_PATHS_PATH = PROJECT_ROOT / "outputs" / "hierarchy_example_reach_paths.parquet"
OUTPUT_MATCHED_REACHES_GPKG_PATH = PROJECT_ROOT / "outputs" / "hierarchy_example_reaches.gpkg"
OUTPUT_MATCHED_REACHES_LAYER = "hierarchy_example_reaches"

WRITE_OUTPUTS = True

print(EXAMPLES_GPKG_PATH)
print(MAIN_CROSSWALK_BEST_PATH)
print(GRDC_CROSSWALK_BEST_PATH)
print(HYDRO_QUALITY_XLSX_PATH)
print(SWORD_PARQUET_DIR)

/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/hierarchy_examples_final.gpkg
/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/crosswalk_best.parquet
/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/grdc/crosswalk_best.parquet
/Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/mask_hydro_quality.xlsx
/Volumes/PhD/SWORD/v17c/beta/parquet


In [3]:
def parse_example_ids(value: object) -> list[int]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    if isinstance(value, (list, tuple, set, np.ndarray)):
        out: list[int] = []
        for item in value:
            out.extend(parse_example_ids(item))
        return out

    if isinstance(value, (int, np.integer)):
        return [int(value)]

    if isinstance(value, float):
        return [int(value)] if np.isfinite(value) else []

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>", "[]"}:
        return []

    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
        except (SyntaxError, ValueError):
            parsed = None
        if parsed is not None:
            return parse_example_ids(parsed)

    return [int(token) for token in re.findall(r"-?\d+", text)]


def parse_semicolon_ints(value: object) -> list[int]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set, np.ndarray)):
        out: list[int] = []
        for item in value:
            out.extend(parse_semicolon_ints(item))
        return out
    text = str(value).strip()
    if not text:
        return []
    return [int(token) for token in re.findall(r"-?\d+", text)]


def parse_station_key_list(value: object) -> list[str]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set, np.ndarray)):
        out: list[str] = []
        for item in value:
            out.extend(parse_station_key_list(item))
        return out
    text = str(value).strip()
    if not text or text == "-":
        return []
    parts = re.split(r"[;,\n]+", text)
    return [part.strip() for part in parts if part.strip() and part.strip() != "-"]


def load_hydro_quality_roles(path: Path, sheet_name: str) -> pd.DataFrame:
    quality = pd.read_excel(path, sheet_name=sheet_name)
    required = {"Example", "Up", "mid", "Down"}
    missing = required - set(quality.columns)
    if missing:
        raise ValueError(f"Hydrograph quality sheet is missing required columns: {sorted(missing)}")

    quality = quality.rename(columns={"Example": "example_id", "Up": "up", "mid": "mid", "Down": "down"}).copy()
    quality["example_id"] = pd.to_numeric(quality["example_id"], errors="coerce").astype("Int64")
    quality = quality.dropna(subset=["example_id"]).copy()
    quality["example_id"] = quality["example_id"].astype(int)
    quality["upstream_station_keys"] = quality["up"].map(parse_station_key_list)
    quality["middle_station_keys"] = quality["mid"].map(parse_station_key_list)
    quality["downstream_station_keys"] = quality["down"].map(parse_station_key_list)

    quality = quality[quality['Peak_match'].isin(['Good', 'Oke'])]

    keep_cols = [
        "example_id",
        "upstream_station_keys",
        "middle_station_keys",
        "downstream_station_keys",
        "up",
        "mid",
        "down"
    ]
    return quality[keep_cols].drop_duplicates(subset=["example_id"]).sort_values("example_id").reset_index(drop=True)


def load_station_examples(path: Path, layer: str) -> gpd.GeoDataFrame:
    stations = gpd.read_file(path, layer=layer)
    if "station_key" not in stations.columns:
        raise ValueError(f"{path} layer '{layer}' is missing station_key.")

    example_source_col = "example_ids" if "example_ids" in stations.columns else "example_id"
    if example_source_col not in stations.columns:
        raise ValueError(f"{path} layer '{layer}' must contain example_ids or example_id.")

    records: list[dict[str, object]] = []
    for row in stations.itertuples(index=False):
        raw_example_value = getattr(row, example_source_col)
        for example_id in parse_example_ids(raw_example_value):
            records.append(
                {
                    "example_id": int(example_id),
                    "station_key": str(getattr(row, "station_key")),
                    "lat": getattr(row, "lat", np.nan),
                    "lon": getattr(row, "lon", np.nan),
                }
            )

    exploded = pd.DataFrame(records)
    if exploded.empty:
        raise ValueError("No example/station assignments were found in the examples file.")

    exploded = exploded.drop_duplicates(subset=["example_id", "station_key"]).reset_index(drop=True)
    return gpd.GeoDataFrame(exploded)


def build_example_station_pairs(stations: pd.DataFrame, hydro_roles: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    records: list[dict[str, object]] = []
    diagnostics: list[dict[str, object]] = []
    hydro_lookup = hydro_roles.set_index("example_id")

    for example_id, group in stations.groupby("example_id", sort=True):
        station_keys_present = set(group["station_key"].dropna().astype(str).tolist())
        if example_id not in hydro_lookup.index:
            raise ValueError(f"Example {example_id} was found in the examples file but not in {HYDRO_QUALITY_XLSX_PATH.name}.")
        role_row = hydro_lookup.loc[example_id]
        upstream_keys = sorted(dict.fromkeys([key for key in role_row["upstream_station_keys"] if key in station_keys_present]))
        middle_keys = sorted(dict.fromkeys([key for key in role_row["middle_station_keys"] if key in station_keys_present]))
        terminal_downstream_keys = sorted(dict.fromkeys([key for key in role_row["downstream_station_keys"] if key in station_keys_present]))
        downstream_keys = sorted(dict.fromkeys(middle_keys + terminal_downstream_keys))
        role_keys = set(role_row["upstream_station_keys"] + role_row["middle_station_keys"] + role_row["downstream_station_keys"])
        missing_role_keys = sorted(role_keys - station_keys_present)
        unexpected_station_keys = sorted(station_keys_present - role_keys)

        diagnostics.append(
            {
                "example_id": int(example_id),
                "station_count": int(len(group)),
                "upstream_station_keys": ";".join(upstream_keys),
                "middle_station_keys": ";".join(middle_keys),
                "downstream_station_keys": ";".join(downstream_keys),
                "has_upstream": bool(upstream_keys),
                "has_downstream": bool(downstream_keys),
                "missing_role_keys": ";".join(missing_role_keys),
                "unexpected_station_keys": ";".join(unexpected_station_keys),
            }
        )

        if upstream_keys and downstream_keys:
            for station_key_up in upstream_keys:
                for station_key_dn in downstream_keys:
                    records.append(
                        {
                            "example_id": int(example_id),
                            "station_key_up": station_key_up,
                            "station_key_dn": station_key_dn,
                        }
                    )
        else:
            records.append(
                {
                    "example_id": int(example_id),
                    "station_key_up": upstream_keys[0] if upstream_keys else pd.NA,
                    "station_key_dn": downstream_keys[0] if downstream_keys else pd.NA,
                }
            )

    pairs = pd.DataFrame(records).sort_values(["example_id", "station_key_up", "station_key_dn"], na_position="last").reset_index(drop=True)
    diagnostic_frame = pd.DataFrame(diagnostics).sort_values("example_id").reset_index(drop=True)
    return pairs, diagnostic_frame


MANUAL_STATION_MATCHES = [
    {
        "station_key": "US:15564860",
        "station_id": "15564860",
        "station_name": "YUKON R AT GALENA AK",
        "lat": 64.7375,
        "lon": -156.891667,
        "country": "US",
        "agency": "usa",
        "reach_id": 81230100071,
        "sword_region": "na",
        "source_file": "manual_example_station_match",
        "distance_m": 0.0,
        "candidate_rank": 1,
        "raw_score": 1.0,
        "ambiguity_penalty": 0.0,
        "total_score": 1.0,
        "distance_score": 1.0,
        "river_name_score": pd.NA,
        "drainage_area_score": pd.NA,
        "second_best_score": pd.NA,
        "candidate_count": 1,
        "score_gap": pd.NA,
        "confidence_class": "manual",
        "review_flag": False,
        "sword_node_id": pd.NA,
        "node_distance_m": pd.NA,
    },
]


def append_manual_station_matches(station_matches: pd.DataFrame, manual_rows: list[dict[str, object]]) -> pd.DataFrame:
    if not manual_rows:
        return station_matches.copy()

    manual = pd.DataFrame(manual_rows)
    combined_columns = list(dict.fromkeys(list(station_matches.columns) + list(manual.columns)))
    station_matches_aligned = station_matches.reindex(columns=combined_columns)
    manual_aligned = manual.reindex(columns=combined_columns)

    combined = pd.concat([station_matches_aligned, manual_aligned], ignore_index=True)
    combined = combined.drop_duplicates(subset=["station_key"], keep="last").reset_index(drop=True)

    if "sword_region" in combined.columns:
        combined["sword_region"] = combined["sword_region"].astype("string").str.lower()
    if "reach_id" in combined.columns:
        combined["reach_id"] = pd.to_numeric(combined["reach_id"], errors="coerce").astype("Int64")

    return combined


def load_combined_best_matches(main_path: Path, grdc_path: Path | None = None) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    main = pd.read_parquet(main_path).copy()
    main["_provider_priority"] = 0
    frames.append(main)

    if grdc_path is not None and grdc_path.exists():
        grdc = pd.read_parquet(grdc_path).copy()
        grdc["_provider_priority"] = 1
        frames.append(grdc)

    combined = pd.concat(frames, ignore_index=True)
    combined = combined.sort_values(["_provider_priority"], kind="mergesort")
    combined = combined.drop_duplicates(subset=["station_key"], keep="first").reset_index(drop=True)

    if "sword_region" in combined.columns:
        combined["sword_region"] = combined["sword_region"].astype("string").str.lower()

    if "reach_id" in combined.columns:
        combined["reach_id"] = pd.to_numeric(combined["reach_id"], errors="coerce").astype("Int64")

    return combined.drop(columns=["_provider_priority"], errors="ignore")


def load_matched_reaches_for_paths(reach_paths: pd.DataFrame, catalog) -> gpd.GeoDataFrame:
    if reach_paths.empty:
        return gpd.GeoDataFrame({"example_id": []}, geometry=gpd.GeoSeries([], crs="EPSG:4326"), crs="EPSG:4326")

    frames: list[gpd.GeoDataFrame] = []
    for region, group in reach_paths.groupby("sword_region", dropna=False):
        if pd.isna(region) or not str(region).strip():
            continue
        reach_ids = sorted(pd.to_numeric(group["reach_id"], errors="coerce").dropna().astype("int64").unique().tolist())
        if not reach_ids:
            continue
        reaches = load_reaches(
            catalog,
            regions=[str(region).lower()],
            reach_ids=reach_ids,
            columns=["reach_id"],
        )
        if reaches.empty:
            continue
        merged = group.merge(reaches, on=["sword_region", "reach_id"], how="left")
        merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=reaches.crs)
        frames.append(merged)

    if not frames:
        return gpd.GeoDataFrame({"example_id": []}, geometry=gpd.GeoSeries([], crs="EPSG:4326"), crs="EPSG:4326")

    crs = frames[0].crs
    return gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), geometry="geometry", crs=crs)


def build_bbox_lookup(matched_reaches: gpd.GeoDataFrame) -> pd.DataFrame:
    if matched_reaches.empty:
        return pd.DataFrame(columns=["example_id", "bbox", "minx", "miny", "maxx", "maxy"])

    records: list[dict[str, object]] = []
    for example_id, group in matched_reaches.groupby("example_id", sort=True):
        working = group.loc[group.geometry.notna()].copy()
        if working.empty:
            bbox = None
            minx = miny = maxx = maxy = pd.NA
        else:
            minx, miny, maxx, maxy = working.total_bounds
            bbox = [float(minx), float(miny), float(maxx), float(maxy)]
        records.append(
            {
                "example_id": int(example_id),
                "bbox": bbox,
                "minx": minx,
                "miny": miny,
                "maxx": maxx,
                "maxy": maxy,
            }
        )
    return pd.DataFrame(records)


In [4]:
station_examples = load_station_examples(EXAMPLES_GPKG_PATH, EXAMPLES_LAYER)
hydro_roles = load_hydro_quality_roles(HYDRO_QUALITY_XLSX_PATH, HYDRO_QUALITY_SHEET)


# example_pairs, pair_diagnostics = build_example_station_pairs(station_examples, hydro_roles)
station_matches = load_combined_best_matches(MAIN_CROSSWALK_BEST_PATH, GRDC_CROSSWALK_BEST_PATH)
station_matches = append_manual_station_matches(station_matches, MANUAL_STATION_MATCHES)
catalog = scan_sword_parquet_dir(SWORD_PARQUET_DIR)

# print("Examples file rows:", len(station_examples))
# print("Unique examples:", station_examples['example_id'].nunique())
# print("Pair rows passed to routing:", len(example_pairs))
# print("Combined station matches:", len(station_matches))
# print("Hydrograph quality role rows:", len(hydro_roles))

display(hydro_roles.head(20))
display(station_matches[station_matches['station_key'].isin([row['station_key'] for row in MANUAL_STATION_MATCHES])])
# display(pair_diagnostics.head(20))
# display(example_pairs.head(20))

example_pairs = hydro_roles[['example_id', 'up', 'down']]
example_pairs = example_pairs.rename(columns={'up': 'station_key_up', 'down': 'station_key_dn'})
display(example_pairs.head(20))


/var/folders/bs/xx1t3fn91jl3k26fn1pndd_c0000gn/T/ipykernel_28285/948640257.py:254: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)
/var/folders/bs/xx1t3fn91jl3k26fn1pndd_c0000gn/T/ipykernel_28285/948640257.py:231: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([station_matches_aligned, manual_aligned], ignore_index=True)


,example_id,upstream_station_keys,middle_station_keys,downstream_station_keys,up,mid,down
0,2,[CL:3179520],[],[CL:3179500],CL:3179520,NaN,CL:3179500
1,3,[BR:3652880],[],[BR:3652890],BR:3652880,NaN,BR:3652890
2,5,[BR:3649421],[],[BR:3649424],BR:3649421,NaN,BR:3649424
3,10,[BR:3635041],[],[BR:3635040],BR:3635041,NaN,BR:3635040
4,11,[BR:3636202],[],[BR:16661000],BR:3636202,NaN,BR:16661000
5,13,[BR:3618721],[BR:3618720],[BR:3618500],BR:3618721,BR:3618720,BR:3618500
6,15,[GF:3512025],[],[GF:3512400],GF:3512025,NaN,GF:3512400
7,21,[TH:2969430],[],[TH:2969090],TH:2969430,NaN,TH:2969090
8,22,[LA:2469260],[KH:2569005],[KH:2569004],LA:2469260,KH:2569005,KH:2569004
9,37,[US:4105810],[],[US:4105800],US:4105810,NaN,US:4105800


,station_key,station_id,station_name,lat,lon,country,agency,drainage_area,river_name,source_function,...,m_end,m_yrs,m_miss,t_start,t_end,t_yrs,lta_discharge,r_volume_yr,r_height_yr,d_yrs
61177,US:15564860,15564860,YUKON R AT GALENA AK,64.7375,-156.891667,US,usa,NaN,<NA>,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,example_id,station_key_up,station_key_dn
0,2,CL:3179520,CL:3179500
1,3,BR:3652880,BR:3652890
2,5,BR:3649421,BR:3649424
3,10,BR:3635041,BR:3635040
4,11,BR:3636202,BR:16661000
5,13,BR:3618721,BR:3618500
6,15,GF:3512025,GF:3512400
7,21,TH:2969430,TH:2969090
8,22,LA:2469260,KH:2569004
9,37,US:4105810,US:4105800


In [5]:
reach_paths, reach_summary = build_example_reach_paths(
    example_pairs,
    station_matches,
    catalog=catalog,
)

print("Routed reach-path rows:", len(reach_paths))
print("Summary rows:", len(reach_summary))
print("Examples with a routed path:", int(reach_summary['route_found'].fillna(False).sum()) if 'route_found' in reach_summary.columns else 0)

display(reach_summary.head(20))
display(reach_paths.head(20))

Routed reach-path rows: 333
Summary rows: 17
Examples with a routed path: 17


,example_id,sword_region,route_status,route_found,all_downstream_reaches_reached,n_reaches_between,upstream_station_keys,downstream_station_keys,upstream_reach_ids,downstream_reach_ids,reaches_between,missing_upstream_station_keys,missing_downstream_station_keys
0,2,sa,ok,True,True,10,CL:3179520,CL:3179500,66220000101,66220000015,66220000101;66220000091;66220000081;6622000007...,,
1,3,sa,ok,True,True,5,BR:3652880,BR:3652890,63580300041,63580100055,63580300041;63580300031;63580300021;6358030001...,,
2,5,sa,ok,True,True,17,BR:3649421,BR:3649424,62495300111,62495100031,62495300111;62495300101;62495300091;6249530008...,,
3,10,sa,ok,True,True,30,BR:3635041,BR:3635040,62265900011,62265300071,62265900011;62265700071;62265700061;6226570005...,,
4,11,sa,ok,True,True,13,BR:3636202,BR:16661000,62252900011,62252500201,62252900011;62252700061;62252700054;6225270004...,,
5,13,sa,ok,True,True,25,BR:3618721,BR:3618500,62284900091,62284500071,62284900091;62284900081;62284900071;6228490006...,,
6,15,sa,ok,True,True,16,GF:3512025,GF:3512400,61660400021,61660100115,61660400021;61660400011;61660500034;6166050002...,,
7,21,as,ok,True,True,15,TH:2969430,TH:2969090,44279300191,44279300041,44279300191;44279300181;44279300171;4427930016...,,
8,22,as,ok,True,True,28,LA:2469260,KH:2569004,44250000171,44230000201,44250000171;44250000151;44250000141;4425000013...,,
9,37,na,ok,True,True,8,US:4105810,US:4105800,81140300071,81140100011,81140300071;81140300061;81140300051;8114030004...,,


,example_id,sword_region,reach_id,path_order,is_upstream_station_reach,is_downstream_station_reach,upstream_station_keys,downstream_station_keys,all_upstream_station_keys,all_downstream_station_keys,upstream_reach_ids,downstream_reach_ids
0,2,sa,66220000101,0,True,False,CL:3179520,,CL:3179520,CL:3179500,66220000101,66220000015
1,2,sa,66220000091,1,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
2,2,sa,66220000081,2,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
3,2,sa,66220000071,3,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
4,2,sa,66220000061,4,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
5,2,sa,66220000051,5,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
6,2,sa,66220000041,6,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
7,2,sa,66220000031,7,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
8,2,sa,66220000025,8,False,False,,,CL:3179520,CL:3179500,66220000101,66220000015
9,2,sa,66220000015,9,False,True,,CL:3179500,CL:3179520,CL:3179500,66220000101,66220000015


In [6]:
matched_reaches = load_matched_reaches_for_paths(reach_paths, catalog)
bbox_lookup = build_bbox_lookup(matched_reaches)

reach_lists = reach_summary.copy()
reach_lists["reach_id_up"] = reach_lists["upstream_reach_ids"].map(parse_semicolon_ints)
reach_lists["reach_id_dn"] = reach_lists["downstream_reach_ids"].map(parse_semicolon_ints)
reach_lists["reaches_between"] = reach_lists["reaches_between"].map(parse_semicolon_ints)
reach_lists = reach_lists.merge(bbox_lookup, on="example_id", how="left")
reach_lists = reach_lists[
    [
        "example_id",
        "reach_id_up",
        "reach_id_dn",
        "reaches_between",
        "n_reaches_between",
        "route_found",
        "bbox",
        "minx",
        "miny",
        "maxx",
        "maxy",
    ]
].sort_values("example_id").reset_index(drop=True)

print("Matched reach geometry rows:", len(matched_reaches))
display(reach_lists.head(20))

Matched reach geometry rows: 1998


,example_id,reach_id_up,reach_id_dn,reaches_between,n_reaches_between,route_found,bbox,minx,miny,maxx,maxy
0,2,[66220000101],[66220000015],"[66220000101, 66220000091, 66220000081, 662200...",10,True,"[-73.16682996325709, -37.575267592593704, -72....",-73.166830,-37.575268,-72.561549,-36.807514
1,3,[63580300041],[63580100055],"[63580300041, 63580300031, 63580300021, 635803...",5,True,"[-41.78643835548293, -21.755194791092304, -41....",-41.786438,-21.755195,-41.295798,-21.593026
2,5,[62495300111],[62495100031],"[62495300111, 62495300101, 62495300091, 624953...",17,True,"[-51.09263969714932, -14.92384596874126, -50.7...",-51.092640,-14.923846,-50.765718,-13.577249
3,10,[62265900011],[62265300071],"[62265900011, 62265700071, 62265700061, 622657...",30,True,"[-65.44782957933442, -9.735969674075898, -63.9...",-65.447830,-9.735970,-63.915586,-8.712895
4,11,[62252900011],[62252500201],"[62252900011, 62252700061, 62252700054, 622527...",13,True,"[-57.063412132962156, -1.437021434539904, -56....",-57.063412,-1.437021,-56.840391,-0.954174
5,13,[62284900091],[62284500071],"[62284900091, 62284900081, 62284900071, 622849...",25,True,"[-61.17757736843309, 1.705133438727843, -60.48...",-61.177577,1.705133,-60.488705,3.497645
6,15,[61660400021],[61660100115],"[61660400021, 61660400011, 61660500034, 616605...",16,True,"[-54.478188271401486, 4.179624634707465, -54.3...",-54.478188,4.179625,-54.385347,4.992391
7,21,[44279300191],[44279300041],"[44279300191, 44279300181, 44279300171, 442793...",15,True,"[101.60551899131036, 17.803417453117895, 102.7...",101.605519,17.803417,102.764446,18.218360
8,22,[44250000171],[44230000201],"[44250000171, 44250000151, 44250000141, 442500...",28,True,"[105.72627364096749, 12.447344874724097, 106.0...",105.726274,12.447345,106.040839,15.155325
9,37,[81140300071],[81140100011],"[81140300071, 81140300061, 81140300051, 811403...",8,True,"[-150.56828578608219, 61.47802183930128, -150....",-150.568286,61.478022,-150.083913,62.222363


In [20]:
reach_lists.loc[reach_lists['example_id'] > 16, 'reaches_between'].explode().to_list()

[44279300191,
 44279300181,
 44279300171,
 44279300161,
 44279300151,
 44279300141,
 44279300131,
 44279300121,
 44279300111,
 44279300101,
 44279300091,
 44279300081,
 44279300071,
 44279300051,
 44279300041,
 44250000171,
 44250000151,
 44250000141,
 44250000131,
 44250000121,
 44250000111,
 44250000101,
 44250000091,
 44250000081,
 44250000071,
 44250000061,
 44250000051,
 44250000041,
 44250000031,
 44250000021,
 44250000011,
 44230000311,
 44230000301,
 44230000291,
 44230000281,
 44230000271,
 44230000261,
 44230000251,
 44230000241,
 44230000231,
 44230000221,
 44230000211,
 44230000201,
 81140300071,
 81140300061,
 81140300051,
 81140300041,
 81140300031,
 81140300021,
 81140300011,
 81140100011,
 81250100171,
 81250100161,
 81250100151,
 81250100141,
 81250100131,
 81250100121,
 81250100111,
 81250100101,
 81250100091,
 81250100081,
 81250100071,
 81250100061,
 81250100051,
 81250100041,
 81250100031,
 81250100021,
 81250100011,
 81241000011,
 81230900011,
 81230700081,
 81230

In [21]:
summary_cols = [
    "example_id",
    "sword_region",
    "route_status",
    "route_found",
    "all_downstream_reaches_reached",
    "n_reaches_between",
    "upstream_station_keys",
    "downstream_station_keys",
    "upstream_reach_ids",
    "downstream_reach_ids",
    "reaches_between",
    "missing_upstream_station_keys",
    "missing_downstream_station_keys",
]
reach_summary_export = reach_summary[[col for col in summary_cols if col in reach_summary.columns]].copy()

if WRITE_OUTPUTS:
    OUTPUT_REACH_LISTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_REACH_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_REACH_PATHS_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_MATCHED_REACHES_GPKG_PATH.parent.mkdir(parents=True, exist_ok=True)

    reach_lists.to_parquet(OUTPUT_REACH_LISTS_PATH, index=False)
    reach_summary_export.to_csv(OUTPUT_REACH_SUMMARY_PATH, index=False)
    reach_paths.to_parquet(OUTPUT_REACH_PATHS_PATH, index=False)

    if OUTPUT_MATCHED_REACHES_GPKG_PATH.exists():
        OUTPUT_MATCHED_REACHES_GPKG_PATH.unlink()
    matched_reaches.to_file(
        OUTPUT_MATCHED_REACHES_GPKG_PATH,
        layer=OUTPUT_MATCHED_REACHES_LAYER,
        driver="GPKG",
    )

    print("Wrote:")
    print(" -", OUTPUT_REACH_LISTS_PATH)
    print(" -", OUTPUT_REACH_SUMMARY_PATH)
    print(" -", OUTPUT_REACH_PATHS_PATH)
    print(" -", OUTPUT_MATCHED_REACHES_GPKG_PATH)
else:
    print("WRITE_OUTPUTS is False. Review tables first, then set it to True and rerun this cell to write outputs.")

Wrote:
 - /Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/hierarchy_example_reach_lists.parquet
 - /Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/hierarchy_example_reach_summary.csv
 - /Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/hierarchy_example_reach_paths.parquet
 - /Users/6256481/Code/river-hierarchy/SWORD_gauge_match/outputs/hierarchy_example_reaches.gpkg


## Notes

- The later bbox builder in `src/gauge_sword_match/example_bboxes.py` reads `hierarchy_example_reach_lists.parquet` and `hierarchy_example_reach_summary.csv`.
- This notebook writes those same files so the downstream bbox workflow remains compatible.
- Upstream/downstream station roles are read from `outputs/mask_hydro_quality.xlsx`, sheet `Hydrograph_quality`.
- `Up` is treated as upstream. `Down` is treated as downstream. `mid` is included with the downstream-side stations when building routed paths.
- The diagnostics table includes `missing_role_keys` and `unexpected_station_keys` so you can catch mismatches between the Excel sheet and the examples GPKG before writing outputs.